# Longitudinal face-aging training

This notebook is the server entry point for the single SD1.5 editing model. It uses one directly sampled diffusion timestep per observation—never a 1,000-step forward corruption loop. Run it in the existing `deep_learning` Conda environment after installing `python -m pip install -e ".[auxiliary,notebooks]"`. ArcFace requires Python 3.11+.

In [ ]:
from pathlib import Path
import torch

from data import build_face_aging_dataloaders
from src.model import build_face_aging_diffusion_bundle
from src.loss import FaceAgingDiffusionLoss
from src.training import TRAIN_AGGING_MODEL, run_training_pipeline_validation

## 1. Data

Only `DATASET_ROOT` needs to change when moving from the sample to the full server dataset. Identity-disjoint splits are produced by the loader. The serious baseline uses 256×256 images, batch size 4, and four accumulation micro-batches (effective batch size 16 on one GPU).

In [ ]:
DATASET_ROOT = Path('/server/path/to/longitudinal_faces')
CHECKPOINT_DIR = Path('/server/path/to/checkpoints/face_aging_minsnr5')
MONITOR_IMAGE = Path('/server/path/to/fixed_monitor_face.jpg')

loaders, data_metadata = build_face_aging_dataloaders(
    DATASET_ROOT,
    image_size=256,
    batch_size=4,
    num_workers=4,
    train_pair_strategy='random_target',
    eval_pair_strategy='all',
    horizontal_flip_prob=0.2,
    pin_memory=True,
    persistent_workers=True,
    train_drop_last=False,  # partial final accumulation is handled exactly
)

## 2. SD1.5 bundle and auxiliary networks

The U-Net receives `[noisy_target_latent; source_latent]` (8 channels). Only LoRA and expanded `conv_in` are trainable. This cell also downloads and freezes `py-feat/arcface_r50` and `iitolstykh/mivolo_v2`. MiVOLO requires reviewed Hugging Face remote code. Both auxiliaries stay resident in FP16 on CUDA; activation checkpointing plus cadence/subsampling reduces VRAM without breaking gradients toward the U-Net. ArcFace weights are restricted to non-commercial research use.

In [ ]:
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
IDENTITY_MODEL_ID = 'py-feat/arcface_r50'
AGE_MODEL_ID = 'iitolstykh/mivolo_v2'

bundle = build_face_aging_diffusion_bundle(
    model_id='stable-diffusion-v1-5/stable-diffusion-v1-5',
    adapter_type='lora',
    rank=16,
    alpha=16,
    dropout=0.0,
    device='cuda',
    dtype=MODEL_DTYPE,
    load_auxiliary_models=True,
    identity_model_id=IDENTITY_MODEL_ID,
    age_model_id=AGE_MODEL_ID,
    auxiliary_dtype=torch.float32,  # ArcFace/MiVOLO normalization paths require FP32
    auxiliary_trust_remote_code=True,  # review/pin age_revision for production
    auxiliary_activation_checkpointing=True,
)

loss_fn = FaceAgingDiffusionLoss(
    scheduler=bundle['scheduler_train'],
    vae=bundle['vae'],
    identity_encoder=bundle['identity_encoder'],
    age_estimator=bundle['age_estimator'],
    identity_weight=0.1,
    age_weight=0.1,
    identity_reference='target',
    age_loss_type='l1',
    auxiliary_every_n_steps=4,
    auxiliary_sample_fraction=0.25,
    auxiliary_max_timestep=400,
    vae_decode_checkpointing=True,
)

## 3. Preflight

This CPU-safe audit checks the uniform timestep and conditioning-dropout distributions. GPU/real-model checks remain explicitly `NOT RUN` unless a server smoke callback is supplied.

In [ ]:
preflight = run_training_pipeline_validation(bundle['scheduler_train'])
preflight

## 4. Train with `TRAIN_AGGING_MODEL`

Parameter groups:

- **Budget:** `num_epochs` is used unless `max_train_steps` is supplied; optimizer steps take precedence.
- **Conservative editing LR:** LoRA `5e-5`, `conv_in` `1e-5`; the lower convolution LR protects spatial structure while new source channels begin learning.
- **Accumulation:** losses are normalized by the actual number of samples, including an incomplete final window.
- **Diffusion:** uniform full-range timesteps and Min-SNR 5 retain generation ability while reducing conflicting timestep gradients. Set `min_snr_gamma=None` for the plain DDPM baseline.
- **CFG preparation:** `conditioning_dropout_prob=0.05` creates 5% text-only, 5% both, 5% image-only dropout, and 85% fully conditioned examples. Identity loss excludes image-dropped samples by default.
- **Posterior policy:** source latent uses the deterministic posterior mean; target latent is sampled during production training.
- **Double prompt:** disabled initially. Set `double_prompt_prob=0.25` only as an explicit FADING-style experiment; the two sequential losses use weights 0.5 + 0.5.
- **Precision:** `auto` prefers BF16, falls back to FP16 on CUDA, and uses GradScaler only for FP16. Trainable parameters remain FP32.
- **Validation/checkpoints:** fixed validation timesteps/noise, best model selected from `val/loss_total`, and exact optimizer/scheduler/RNG resume.

In [ ]:
training_state = TRAIN_AGGING_MODEL(
    bundle=bundle,
    loss_fn=loss_fn,
    train_loader=loaders['train'],
    val_loader=loaders['val'],

    num_epochs=10,
    max_train_steps=None,
    lr_lora=5e-5,
    lr_conv_in=1e-5,
    weight_decay=1e-2,
    conv_in_weight_decay=1e-2,
    warmup_ratio=0.05,
    min_lr_ratio=0.10,
    grad_accum_steps=4,
    max_grad_norm=1.0,

    timestep_sampling='uniform',
    min_train_timestep=0,
    max_train_timestep=None,  # full scheduler range
    min_snr_gamma=5.0,
    auxiliary_max_timestep=400,
    conditioning_dropout_prob=0.05,
    identity_loss_on_image_dropped_samples=False,
    sample_source_posterior=False,
    sample_target_posterior=True,
    noise_offset=0.0,
    double_prompt_prob=0.0,

    amp_enabled=True,
    amp_dtype='auto',
    gradient_checkpointing=True,
    enable_xformers=True,

    checkpoint_dir=CHECKPOINT_DIR,
    monitor='val/loss_total',
    monitor_mode='min',
    validate_every_epochs=1,
    deterministic_validation=True,
    validation_seed=2026,
    save_epoch_checkpoints=True,
    max_epoch_checkpoints=5,
    sample_every_epochs=1,
    sample_fn=None,  # None activates built-in inference monitoring below
    monitoring_image=MONITOR_IMAGE,
    monitoring_target_age=[30, 40, 50, 65],  # ordered sweep saved inside each epoch folder
    monitoring_source_age=None,
    monitoring_use_inverse_diffusion=True,  # False selects direct mode
    monitoring_num_inference_steps=30,
    monitoring_strength=0.45,
    monitoring_seed=2026,  # fixed across epochs

    seed=42,
    deterministic=False,
    device='auto',
)